# GB_code — Minimal Working Example

This notebook demonstrates how to construct a grain boundary structure using the GB_code Python API and export it to pymatgen, ASE, LAMMPS and VASP formats.

We will build a **Sigma-5 [1,0,0] (0,3,1) symmetric tilt** grain boundary in FCC aluminium.

## 0. Installation

```bash
pip install ".[all]"   # installs GB_code + pymatgen + ASE
```

## 1. Find available CSL boundaries

Before building a GB we need to know which sigma values and GB planes are available for our chosen rotation axis. The `csl_generator` module provides this.

In [1]:
from gb_code import csl_generator as cslgen
import numpy as np

# Choose a rotation axis
axis = [1, 0, 0]

# List all sigma values up to 50
print("Available CSL boundaries for axis", axis)
print("=" * 45)
cslgen.print_list(axis, limit=50)

Available CSL boundaries for axis [1, 0, 0]
Sigma:     1  Theta:   0.00 
Sigma:     5  Theta:  36.87 
Sigma:    13  Theta:  22.62 
Sigma:    17  Theta:  28.07 
Sigma:    25  Theta:  16.26 
Sigma:    29  Theta:  43.60 
Sigma:    37  Theta:  18.92 
Sigma:    41  Theta:  12.68 


In [2]:
# Pick Sigma 5 and list available GB planes for FCC
basis = 'fcc'

# Sigma 5 for [1,0,0] corresponds to m=2, n=1 (theta ~36.87 deg)
m, n = 2, 1
sigma = cslgen.get_cubic_sigma(axis, m, n)
theta_deg = np.degrees(cslgen.get_cubic_theta(axis, m, n))
print(f"Sigma = {sigma}, theta = {theta_deg:.2f} degrees")
print()

# List GB planes (lim controls the maximum Miller index)
print("Available GB planes for Sigma-5 [1,0,0] FCC:")
print("-" * 70)
cslgen.print_list_GB_Planes(axis, basis, m, n, lim=2)

Sigma = 5, theta = 53.13 degrees

Available GB planes for Sigma-5 [1,0,0] FCC:
----------------------------------------------------------------------


[0 1 2]                [ 0 -1  2]             Symmetric Tilt         40        
[ 0 -1 -2]             [ 0  1 -2]             Symmetric Tilt         40        
[ 0  2 -1]             [0 2 1]                Symmetric Tilt         40        
[-1  0  0]             [-1  0  0]             Twist                  40        
[1 0 0]                [1 0 0]                Twist                  40        
[ 0 -2  1]             [ 0 -2 -1]             Symmetric Tilt         40        
[-1 -1 -2]             [-1  1 -2]             Mixed                  60        


[-1  1  2]             [-1 -1  2]             Mixed                  60        
[ 0  1 -3]             [ 0  3 -1]             Symmetric Tilt         20        
[0 3 1]                [0 1 3]                Symmetric Tilt         20        
[1 1 2]                [ 1 -1  2]             Mixed                  60        
[ 1 -1 -2]             [ 1  1 -2]             Mixed                  60        
[ 0 -1  3]             [ 0 -3  1]             Symmetric Tilt         20        
[ 0 -3 -1]             [ 0 -1 -3]             Symmetric Tilt         20        


[-1  2 -1]             [-1  2  1]             Mixed                  60        
[-1 -2  1]             [-1 -2 -1]             Mixed                  60        
[ 1 -2  1]             [ 1 -2 -1]             Mixed                  60        
[ 1  2 -1]             [1 2 1]                Mixed                  60        
[-1  3  1]             [-1  1  3]             Mixed                  220       
[-1 -1  3]             [-1 -3  1]             Mixed                  220       
[ 1 -3 -1]             [ 1 -1 -3]             Mixed                  220       
[1 3 1]                [1 1 3]                Mixed                  220       


[ 1 -1  3]             [ 1 -3  1]             Mixed                  220       
[ 1  1 -3]             [ 1  3 -1]             Mixed                  220       
[-1  1 -3]             [-1  3 -1]             Mixed                  220       
[-1 -3 -1]             [-1 -1 -3]             Mixed                  220       
[0 0 5]                [ 0 -4  3]             Tilt                   200       
[ 0 -3  4]             [ 0 -5  0]             Tilt                   200       
[-2  1  2]             [-2 -1  2]             Mixed                  360       
[-2  2 -1]             [-2  2  1]             Mixed                  360       


...

[ 1 -3  4]             [ 1 -5  0]             Mixed                  1300      
[-1  0  5]             [-1 -4  3]             Mixed                  1300      
[-1 -4 -3]             [-1  0 -5]             Mixed                  1300      
[-1  0 -5]             [-1  4 -3]             Mixed                  1300      
[-2 -3 -1]             [-2 -1 -3]             Mixed                  140       
[1 0 5]                [ 1 -4  3]             Mixed                  1300      
[ 1 -4 -3]             [ 1  0 -5]             Mixed                  1300      


[1 5 0]                [1 3 4]                Mixed                  1300      
[ 1  3 -4]             [1 5 0]                Mixed                  1300      
[1 4 3]                [1 0 5]                Mixed                  1300      
[ 1  0 -5]             [ 1  4 -3]             Mixed                  1300      
[-2  4  3]             [-2  0  5]             Mixed                  5800      
[-2  3 -4]             [-2  5  0]             Mixed                  5800      
[-2  5  0]             [-2  3  4]             Mixed                  5800      


[-2  0 -5]             [-2  4 -3]             Mixed                  5800      
[ 2 -5  0]             [ 2 -3 -4]             Mixed                  5800      
[ 2 -3  4]             [ 2 -5  0]             Mixed                  5800      
[ 2  0 -5]             [ 2  4 -3]             Mixed                  5800      
[2 4 3]                [2 0 5]                Mixed                  5800      
[-1 -6 -2]             [-1 -2 -6]             Mixed                  820       
[-1 -2  6]             [-1 -6  2]             Mixed                  820       
[-2  0  5]             [-2 -4  3]             Mixed                  5800      


[-2 -5  0]             [-2 -3 -4]             Mixed                  5800      
[-1  2 -6]             [-1  6 -2]             Mixed                  820       
[-1  6  2]             [-1  2  6]             Mixed                  820       
[-2 -3  4]             [-2 -5  0]             Mixed                  5800      
[-2 -4 -3]             [-2  0 -5]             Mixed                  5800      
[ 1 -2  6]             [ 1 -6  2]             Mixed                  820       
[ 1  2 -6]             [ 1  6 -2]             Mixed                  820       


[1 6 2]                [1 2 6]                Mixed                  820       
[ 1 -6 -2]             [ 1 -2 -6]             Mixed                  820       
[ 2  3 -4]             [2 5 0]                Mixed                  5800      
[2 5 0]                [2 3 4]                Mixed                  5800      
[ 2 -4 -3]             [ 2  0 -5]             Mixed                  5800      
[2 0 5]                [ 2 -4  3]             Mixed                  5800      


## 2. Build the grain boundary

We pick the **(0,3,1)** symmetric tilt plane. The workflow is:

1. `ParseGB` — set up axis, basis, lattice parameter, m, n, and GB plane
2. `CSL_Bicrystal_Atom_generator` — populate the two grain unit cells
3. `build` — expand to supercell dimensions and (optionally) remove overlapping atoms

In [3]:
from gb_code.gb_generator import GB_character

gb = GB_character()

# Step 1: Parse GB parameters
gb.ParseGB(
    axis=[1, 0, 0],
    basis='fcc',
    LatP=4.05,       # Al lattice parameter in Angstrom
    m=2,
    n=1,
    gb=[0, 3, 1]     # symmetric tilt plane
)

# Step 2: Generate bicrystal unit cells
gb.CSL_Bicrystal_Atom_generator()

# Step 3: Build supercell
# - overlap=0.0 means no atom removal
# - dim=[2,1,1] doubles the cell along the GB normal to reduce image interaction
gb.build(overlap=0.0, whichG='g1', dim=[2, 1, 1])

print(gb)

GB_character(sigma=5, axis=[1 0 0], plane=[0 3 1], n_grain1=20, n_grain2=20)


In [4]:
# Inspect the geometry
info = gb.get_grain_info()
for key, val in info.items():
    print(f"{key:>30s}: {val}")

                     gb_normal: x
           gb_plane_x_angstrom: 12.807224523681937
     periodic_image_x_angstrom: 25.614449047363873
                      n_grain1: 20
                      n_grain2: 20
                          cell: [[25.61444905  0.          0.        ]
 [ 0.          4.05        0.        ]
 [ 0.          0.          6.40361226]]
                         sigma: 5
                          axis: [1, 0, 0]
                       gbplane: [0, 3, 1]
                     theta_rad: 0.9272952180016122


## 3. Export to pymatgen

In [5]:
structure = gb.to_pymatgen(element='Al')

print(f"Formula:      {structure.composition}")
print(f"Num sites:    {structure.num_sites}")
print(f"Cell (Ang):   a={structure.lattice.a:.3f}  b={structure.lattice.b:.3f}  c={structure.lattice.c:.3f}")
print()

# Grain labels are stored as site properties
grain_ids = structure.site_properties['grain_id']
n_g1 = sum(1 for g in grain_ids if g == 1)
n_g2 = sum(1 for g in grain_ids if g == 2)
print(f"Grain 1 atoms: {n_g1}")
print(f"Grain 2 atoms: {n_g2}")
structure

Formula:      Al40
Num sites:    40
Cell (Ang):   a=25.614  b=4.050  c=6.404

Grain 1 atoms: 20
Grain 2 atoms: 20


Structure Summary
Lattice
    abc : 25.614449047363873 4.05 6.403612261840968
 angles : 90.0 90.0 90.0
 volume : 664.3012500000001
      A : np.float64(25.614449047363873) np.float64(0.0) np.float64(0.0)
      B : np.float64(0.0) np.float64(4.05) np.float64(0.0)
      C : np.float64(0.0) np.float64(0.0) np.float64(6.403612261840968)
    pbc : True True True
PeriodicSite: Al (13.45, 2.025, 4.483) [0.525, 0.5, 0.7]
PeriodicSite: Al (14.09, 0.0, 2.561) [0.55, 0.0, 0.4]
PeriodicSite: Al (15.37, 0.0, 5.123) [0.6, 0.0, 0.8]
PeriodicSite: Al (16.01, 2.025, 3.202) [0.625, 0.5, 0.5]
PeriodicSite: Al (17.29, 2.025, 5.763) [0.675, 0.5, 0.9]
PeriodicSite: Al (17.93, 0.0, 3.842) [0.7, 0.0, 0.6]
PeriodicSite: Al (12.81, 0.0, 0.0) [0.5, 0.0, 0.0]
PeriodicSite: Al (14.73, 2.025, 0.6404) [0.575, 0.5, 0.1]
PeriodicSite: Al (16.65, 0.0, 1.281) [0.65, 0.0, 0.2]
PeriodicSite: Al (18.57, 2.025, 1.921) [0.725, 0.5, 0.3]
PeriodicSite: Al (19.85, 2.025, 4.483) [0.775, 0.5, 0.7]
PeriodicSite: Al (20.49, 0.0, 2.

## 4. Export to ASE

In [6]:
atoms = gb.to_ase(element='Al')

print(f"Formula:   {atoms.get_chemical_formula()}")
print(f"Num atoms: {len(atoms)}")
print(f"Cell:")
print(atoms.cell[:])
print()
print(f"GB plane at x = {atoms.info['gb_plane_x']:.3f} Ang")
print(f"Sigma = {atoms.info['sigma']}")
print()

# Grain IDs are in tags and in arrays['grain_id']
tags = atoms.get_tags()
print(f"Grain 1 atoms (tag=1): {np.sum(tags == 1)}")
print(f"Grain 2 atoms (tag=2): {np.sum(tags == 2)}")
atoms.write('sigma5_031_Al.xyz')

Formula:   Al40
Num atoms: 40
Cell:
[[25.61444905  0.          0.        ]
 [ 0.          4.05        0.        ]
 [ 0.          0.          6.40361226]]

GB plane at x = 12.807 Ang
Sigma = 5

Grain 1 atoms (tag=1): 20
Grain 2 atoms (tag=2): 20


## 5. Write to LAMMPS / VASP files

In [7]:
# Write LAMMPS data file
gb.write_lammps('sigma5_031_Al.lammps')
print("Wrote sigma5_031_Al.lammps")

# Write VASP POSCAR
gb.write_vasp('POSCAR_sigma5_031_Al')
print("Wrote POSCAR_sigma5_031_Al")

Wrote sigma5_031_Al.lammps
Wrote POSCAR_sigma5_031_Al


## 6. With overlap removal

When atoms from the two grains overlap near the GB plane, you can remove them by setting `overlap > 0`. This is a fraction of the lattice parameter.

In [8]:
gb2 = GB_character()
gb2.ParseGB([1, 0, 0], 'fcc', 4.05, 2, 1, [0, 3, 1])
gb2.CSL_Bicrystal_Atom_generator()

# Remove overlapping atoms from grain 1 within 0.3 * LatP
gb2.build(overlap=0.3, whichG='g1', dim=[2, 1, 1])

print(gb2)
print()

info2 = gb2.get_grain_info()
print(f"Grain 1: {info2['n_grain1']} atoms")
print(f"Grain 2: {info2['n_grain2']} atoms")

<<------ 0 atoms are being removed! ------>>
GB_character(sigma=5, axis=[1 0 0], plane=[0 3 1], n_grain1=20, n_grain2=20)

Grain 1: 20 atoms
Grain 2: 20 atoms


## 7. In-plane rigid body translations

To explore microscopic degrees of freedom, scan rigid body translations on the GB plane. This generates multiple candidate structures for energy minimisation.

In [9]:
from gb_code.inplane_shift import generate_shifts, apply_shift, get_all_shifted_structures

# Build a base GB
gb3 = GB_character()
gb3.ParseGB([1, 0, 0], 'fcc', 4.05, 2, 1, [0, 3, 1])
gb3.CSL_Bicrystal_Atom_generator()
gb3.build(overlap=0.3, whichG='g1', dim=[2, 1, 1])

# Generate shift vectors on a 10x5 grid (50 shifts)
shifts = generate_shifts(gb3, a=10, b=5)
print(f"Number of shift vectors: {len(shifts)}")
print(f"First few shifts (lattice units):")
for s in shifts[:5]:
    print(f"  {s}")

<<------ 0 atoms are being removed! ------>>
Number of shift vectors: 50
First few shifts (lattice units):
  [0. 0. 0.]
  [0.  0.2 0. ]
  [0.  0.4 0. ]
  [0.  0.6 0. ]
  [0.  0.8 0. ]


In [10]:
# Apply a single shift and get a pymatgen Structure
shifted_struct = apply_shift(gb3, shifts[7], output='pymatgen', element='Al')
print(f"Shifted structure: {shifted_struct.num_sites} sites")
print()

# Or get all shifted structures at once (as ASE Atoms)
all_atoms = get_all_shifted_structures(gb3, a=5, b=3, output='ase', element='Al')
print(f"Generated {len(all_atoms)} shifted structures")
print(f"Atoms per structure: {len(all_atoms[0])}")

Shifted structure: 40 sites

Generated 15 shifted structures
Atoms per structure: 40


## Summary

| Step | Method | Description |
|------|--------|-------------|
| 1 | `cslgen.print_list(axis, limit)` | List sigma values for a rotation axis |
| 2 | `cslgen.print_list_GB_Planes(axis, basis, m, n)` | List available GB planes |
| 3 | `gb.ParseGB(...)` | Configure the GB |
| 4 | `gb.CSL_Bicrystal_Atom_generator()` | Build bicrystal unit cells |
| 5 | `gb.build(overlap, whichG, dim)` | Expand supercell, remove overlaps |
| 6 | `gb.to_pymatgen()` / `gb.to_ase()` | Export to pymatgen or ASE |
| 7 | `gb.write_lammps()` / `gb.write_vasp()` | Write LAMMPS/VASP files |
| 8 | `generate_shifts()` / `apply_shift()` | In-plane rigid body translations |